# Generate assignment samples

| Step | What to do |
|---|---|
| 1 | Fill the settings cell below: code URL **or** existing code folder |
| 2 | Mount your Drive and run setup |
| 3 | Run the remaining cells in order |

[Full settings and folder layout](../START_HERE.md#notebook-settings) ? [English checkpoints](https://drive.google.com/file/d/1PkGkmCfh1C-CWESDjNyl2X6fdpA-Glz-/view?usp=sharing) ? [Telugu checkpoints](https://drive.google.com/file/d/19iQEstcLrXOf2nZ_M55uACsjxP0Oa9xk/view?usp=sharing)

**BACKUP_DIR is the extracted backup folder path, not a sharing URL.** Download permissions are not independently verified. GPU is recommended; CPU inference works.

In [ ]:
# Read START_HERE.md (Notebook settings) for asset links and how to find these values.
# Choose ONE code source below; these are not both required.
REPO_URL = "https://github.com/pragati-jain340/fastspeech2-english-telugu.git"  # Public project repository
EXISTING_REPO = ""  # Alternative: existing code folder, e.g. /content/drive/MyDrive/my-fast-speech-repo
# Your chosen persistent storage folder; the default is fine and is created if absent.
DATA_ROOT = "/content/drive/MyDrive/FastSpeech2_Assignment"
MOUNT_DRIVE = True
BACKUP_DIR = ""  # Path to downloaded/extracted weight backups, NOT a sharing URL; see START_HERE.md (Notebook settings)
VARIANT = "frame_prosody"
DEVICE = "auto"
OUTPUT = DATA_ROOT + "/Results/" + VARIANT


In [ ]:
from pathlib import Path
import subprocess, sys, os
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if EXISTING_REPO:
    checkout = Path(EXISTING_REPO)
else:
    if not REPO_URL:
        raise ValueError("Set REPO_URL to the submitted GitHub clone URL, or EXISTING_REPO to your downloaded code folder. See START_HERE.md for settings.")
    checkout = Path("/content/fastspeech2-assignment")
    if not checkout.exists():
        subprocess.run(["git", "clone", REPO_URL, str(checkout)], check=True)
PROJECT = checkout / "project" if (checkout / "project").is_dir() else checkout
if not (PROJECT / "shared/cli.py").is_file():
    raise FileNotFoundError(f"Not the completed assignment repository: {PROJECT}")
os.chdir(PROJECT)
import torch, torchaudio
print("Existing PyTorch:", torch.__version__, "torchaudio:", torchaudio.__version__)
constraints = Path("/content/fs2-runtime-constraints.txt")
constraints.write_text(f"torch=={torch.__version__}\ntorchaudio=={torchaudio.__version__}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt",
                "--constraint", str(constraints)], check=True)
ROOT = Path(DATA_ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True, cwd=PROJECT)
print("Project:", PROJECT, "Persistent data:", ROOT, "GPU:", torch.cuda.is_available())


**Required:** both acoustic checkpoint backups. Raw training data is not needed.

Run import ? generation ? playback ? ZIP export. The same pretrained HiFi-GAN is cached on Drive. Original sentences and separate spoken-number expansions come from test_sentences.json.

In [ ]:
if BACKUP_DIR:
    if not Path(BACKUP_DIR).is_dir():
        raise ValueError("BACKUP_DIR must be an existing extracted backup folder. Download links and folder layout are in START_HERE.md.")
    run("scripts/import_assets.py", "--source", BACKUP_DIR, "--destination", ROOT)
else:
    print("Checkpoint location:", ROOT / "runs", "if present. Set BACKUP_DIR to import supplied weights.")


In [ ]:
run("scripts/evaluate.py", "--root", ROOT, "--output", OUTPUT, "--variant", VARIANT, "--device", DEVICE)

In [ ]:
from IPython.display import Audio, display
for language in ("English", "Telugu"):
    for wav in sorted((Path(OUTPUT) / language / "test_generated_audios").glob("*.wav")):
        print(language, wav.stem)
        display(Audio(filename=str(wav)))


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/assignment_outputs", "zip", OUTPUT)
files.download(archive)
